# Tutorial 02 — Exploring all the fields

P3D writes ~30 movie variables per timestep — three components each of B, E, J for each species, plus densities and the full pressure tensor. This notebook:

1. Lists what's available via `m.movie_vars`.
2. Reads everything in one call using `get_fields('all', time=t)`.
3. Builds a 3×3 panel showing B, E, J vector components.

If you have not done tutorial 01, do that first — it explains the basics of `Movie` and `ims`.

In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath('..'))

import matplotlib.pyplot as plt
import py3d
from data_path import DATA_DIR, PARAM_FILE, NAME_STYLE, require_data_dir

require_data_dir()

## What variables are in this run?

`m.movie_vars` is a list of names in the order they were written to the binary movie files. The order matters for the on-disk format but you'll mostly use it as a lookup.

In [ ]:
m = py3d.Movie(
    num=0,
    path=DATA_DIR,
    param_file=PARAM_FILE,
    name_style=NAME_STYLE,
    interactive=False,
)

print(f'{len(m.movie_vars)} variables:')
print('  ' + ', '.join(m.movie_vars))

Group them mentally:

- **Bulk fields:** `rho`, `jx`, `jy`, `jz`, `bx`, `by`, `bz`, `ex`, `ey`, `ez`
- **Electrons:** `ne`, `jex`, `jey`, `jez`, plus the symmetric pressure tensor `pexx, peyy, pezz, pexy, peyz, pexz`
- **Ions:** `ni`, `jix`, `jiy`, `jiz`, plus `pixx, piyy, pizz, pixy, piyz, pixz`

## Read everything at one timestep

Passing `'all'` to `get_fields` reads every variable. The result is the same dict you saw in tutorial 01, just with more keys.

In [ ]:
time_index = m.ntimes // 2
d = m.get_fields('all', time=time_index)

print(f'Read {len(m.movie_vars)} fields at frame {time_index} '
      f'(t = {d["tt"][-1]:.2f} 1/Omega_ci)')
print(f'Each field has shape {d["bz"].shape}')

## A 3×3 panel of B, E, J

Same `ims` call as tutorial 01, just looped. Putting `cont=True` only on the top-left panel keeps the flux contours as a single visual reference without cluttering every subplot.

In [ ]:
panels = [
    ['bx', 'by', 'bz'],
    ['ex', 'ey', 'ez'],
    ['jx', 'jy', 'jz'],
]

fig, axes = plt.subplots(
    nrows=3, ncols=3, figsize=(14, 8), sharex=True, sharey=True
)

for i, row in enumerate(panels):
    for j, var in enumerate(row):
        ax = axes[i, j]
        py3d.ims(d, var, ax=ax, cbar=True,
                 cont=(i == 0 and j == 0),
                 cmap='RdBu_r')
        ax.set_title(var)
        if i < 2:
            ax.set_xlabel('')
        if j > 0:
            ax.set_ylabel('')

fig.suptitle(f'B, E, J at frame {time_index} (t = {d["tt"][-1]:.2f} 1/Omega_ci)',
             fontsize=14)
fig.tight_layout()

## A quick survey of amplitudes

Once everything is in `d`, a one-liner over `m.movie_vars` is often the fastest way to see which fields actually carry signal.

In [ ]:
for var in m.movie_vars:
    arr = d[var]
    print(f'  {var:>5s}  max |.| = {abs(arr).max():.4f}   std = {arr.std():.4f}')

## What you just learned

- `m.movie_vars` — list of variable names available in this run.
- `m.get_fields('all', time=t)` — load every variable at one timestep in a single call.
- The same `py3d.ims(d, var, ax=ax, ...)` works for any 2D field; loop it for multi-panel plots.

**Next:** [Tutorial 03 — Time evolution](03_time_evolution.ipynb) loops over the available frames to build a reconnection-rate time series.